#**Clustering de sentiments sur des critiques de films (IMDb)**

Ce projet vise à appliquer un algorithme de clustering non supervisé (K-Means) sur des critiques de films, en combinant des caractéristiques textuelles (TF-IDF) et des indicateurs de sentiment, afin d’analyser automatiquement les opinions positives et négatives.

#Installation des dépendances nécessaires
Cette cellule installe les bibliothèques nécessaires à l’analyse de sentiment (TextBlob et VADER).
À exécuter une seule fois dans l’environnement local.

In [ ]:
!pip install textblob vaderSentiment
!python -m textblob.download_corpora

#Importation des bibliothèques

Cette cellule importe toutes les bibliothèques Python utilisées

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

#Vérification des bibliothèques d’analyse de sentiment

Cette cellule vérifie si TextBlob et VADER sont bien installées et activables.
Si ce n’est pas le cas, le code continuera avec des valeurs par défaut.

In [ ]:
try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
except:
    TEXTBLOB_AVAILABLE = False

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    VADER_AVAILABLE = True
except:
    VADER_AVAILABLE = False

#Configuration de l’affichage graphique

Cette cellule définit un style graphique uniforme pour toutes les visualisations.

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

#Chargement des données IMDb

Cette cellule définit les chemins locaux vers les données IMDb et charge les critiques positives et négatives à partir de fichiers texte.

In [ ]:
BASE_DIR = r"C:\ML\aclImdb"

train_pos_path = os.path.join(BASE_DIR, "train", "pos")
train_neg_path = os.path.join(BASE_DIR, "train", "neg")

def load_reviews(folder, sentiment):
    reviews = []
    if not os.path.exists(folder):
        return pd.DataFrame(columns=["review", "sentiment"])

    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            with open(os.path.join(folder, filename), "r", encoding="utf-8", errors="ignore") as f:
                reviews.append(f.read())

    return pd.DataFrame({
        "review": reviews,
        "sentiment": sentiment
    })

df_pos = load_reviews(train_pos_path, "positive")
df_neg = load_reviews(train_neg_path, "negative")
df_train = pd.concat([df_pos, df_neg]).reset_index(drop=True)

#Prétraitement avancé du texte

Cette cellule nettoie le texte :
* mise en minuscules,
* suppression du HTML,
* conservation des négations,
* suppression des caractères inutiles.

In [ ]:
POSITIVE_WORDS = {
    'good','great','excellent','amazing','wonderful','fantastic','love','best',
    'perfect','brilliant','outstanding','awesome','incredible','recommended'
}

NEGATIVE_WORDS = {
    'bad','terrible','awful','worst','hate','horrible','boring','disappointing',
    'waste','stupid','weak','mess','annoying','garbage'
}

def advanced_preprocess(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r'[^a-zA-Z\s!?]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_train['clean_review'] = df_train['review'].apply(advanced_preprocess)

#Création des caractéristiques de sentiment

Cette cellule extrait plusieurs indicateurs :

* nombre de mots positifs et négatifs,
* longueur du texte,
* ponctuation émotionnelle,
* scores TextBlob et VADER.

In [ ]:
def count_sentiment_words(text):
    words = set(text.split())
    return len(words & POSITIVE_WORDS), len(words & NEGATIVE_WORDS)

df_train[['pos_word_count','neg_word_count']] = df_train['clean_review'].apply(
    lambda x: pd.Series(count_sentiment_words(x))
)

df_train['sentiment_ratio'] = (
    df_train['pos_word_count'] - df_train['neg_word_count']
) / (df_train['pos_word_count'] + df_train['neg_word_count'] + 1)

df_train['review_length'] = df_train['clean_review'].str.len()
df_train['word_count'] = df_train['clean_review'].str.split().str.len()
df_train['exclamation_count'] = df_train['review'].str.count('!')
df_train['question_count'] = df_train['review'].str.count('\?')

#Analyse de sentiment avec TextBlob et VADER

Cette cellule ajoute des scores de polarité et d’émotion issus de bibliothèques NLP.

In [ ]:
if TEXTBLOB_AVAILABLE:
    df_train['polarity'] = df_train['clean_review'].apply(lambda x: TextBlob(x).sentiment.polarity)
    df_train['subjectivity'] = df_train['clean_review'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
else:
    df_train['polarity'] = 0
    df_train['subjectivity'] = 0

if VADER_AVAILABLE:
    vader = SentimentIntensityAnalyzer()
    scores = df_train['clean_review'].apply(vader.polarity_scores)
    df_train['vader_compound'] = scores.apply(lambda x: x['compound'])
    df_train['vader_positive'] = scores.apply(lambda x: x['pos'])
    df_train['vader_negative'] = scores.apply(lambda x: x['neg'])
    df_train['vader_neutral'] = scores.apply(lambda x: x['neu'])
else:
    df_train[['vader_compound','vader_positive','vader_negative','vader_neutral']] = 0

#Vectorisation TF-IDF

Cette cellule transforme le texte en vecteurs numériques basés sur l’importance des mots.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=3,
    max_df=0.6,
    stop_words='english',
    ngram_range=(1,2),
    sublinear_tf=True
)

X_tfidf = vectorizer.fit_transform(df_train['clean_review'])

#Fusion des caractéristiques

Cette cellule combine les vecteurs TF-IDF avec les caractéristiques de sentiment normalisées.

In [ ]:
sentiment_cols = [
    'pos_word_count','neg_word_count','sentiment_ratio',
    'review_length','word_count','exclamation_count','question_count',
    'polarity','subjectivity','vader_compound',
    'vader_positive','vader_negative','vader_neutral'
]

X_sentiment = df_train[sentiment_cols].values
scaler = StandardScaler()
X_sentiment_scaled = scaler.fit_transform(X_sentiment)

from scipy.sparse import hstack, csr_matrix
X_combined = hstack([X_tfidf, csr_matrix(X_sentiment_scaled)])

#Réduction de dimension

Cette cellule réduit la dimension des données tout en conservant l’essentiel de l’information.

In [ ]:
svd = TruncatedSVD(n_components=150, random_state=42)
X_reduced = svd.fit_transform(X_combined)

pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_reduced)

#Détermination du nombre optimal de clusters

Cette cellule utilise la méthode du coude et le score de silhouette.

In [ ]:
K_range = range(2,11)
inertias = []
silhouette_scores = []

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_reduced)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_reduced, labels))

#Application du K-Means

Cette cellule applique le clustering final avec K = 2.

In [ ]:
kmeans = KMeans(n_clusters=2, n_init=20, max_iter=500, random_state=42)
df_train['cluster'] = kmeans.fit_predict(X_reduced)

#Évaluation des performances

Cette cellule mesure la qualité du clustering par plusieurs métriques.

In [ ]:
y_true = df_train['sentiment'].map({'positive':1,'negative':0}).values
labels = df_train['cluster'].values

sil_score = silhouette_score(X_reduced, labels)
ari = adjusted_rand_score(y_true, labels)
nmi = normalized_mutual_info_score(y_true, labels)

cm = confusion_matrix(y_true, labels)
accuracy = (cm[0,0] + cm[1,1]) / cm.sum()

print(classification_report(y_true, labels, target_names=['Negative','Positive']))

#Analyse des mots discriminants

Cette cellule identifie les mots les plus représentatifs de chaque cluster.

In [ ]:
feature_names = vectorizer.get_feature_names_out()

def get_top_keywords(cluster_id, labels, tfidf_matrix, top_n=15):
    idx = np.where(labels == cluster_id)[0]
    mean_tfidf = tfidf_matrix[idx].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-top_n:][::-1]
    return [(feature_names[i], mean_tfidf[i]) for i in top_idx]

#Conclusion

Cette dernière cellule résume les résultats obtenus et confirme la réussite de l’analyse.

In [ ]:
print("Analyse de clustering terminée")
print(f"Accuracy : {accuracy*100:.2f}%")
print(f"ARI : {ari:.4f}")